# 05 — Lightning full-parameter Text2SQL SFT design

This notebook is deliberately design-only. Full SFT updates all 30B Lightning
parameters; sparse activation reduces per-token compute but not optimizer state.
A normal one- or two-GPU Brev instance cannot run honest full-parameter AdamW.

The guarded driver uses NVIDIA's shipped Lightning full-SFT recipe and is gated
to the verified 16×H100 80 GB topology (TP2/EP8) with at least 1,200 GiB aggregate
VRAM. Nano full SFT is outside this repository's workshop contract.


In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
from nemotron_ft_lab.model_profiles import LIGHTNING35_ADVANCED as PROFILE

ARTIFACTS_DIR = Path(os.environ.get('NEMOTRON_ARTIFACTS_DIR', ROOT / 'artifacts')).expanduser().resolve()
DATA_DIR = ARTIFACTS_DIR / 'data/bird-text2sql/profiles' / PROFILE.name
BASELINE_PATH = ARTIFACTS_DIR / 'evaluation' / PROFILE.artifact_slug / 'baseline_local_bf16_text2sql.json'
MEGATRON_BASE = Path('/workspace/storage/checkpoints') / PROFILE.megatron_checkpoint_name
FULL_ROOT = Path('/workspace/storage/checkpoints/bird-text2sql-lightning35-full')
FULL_HF = Path('/workspace/storage/checkpoints/bird-text2sql-lightning35-full-hf')
DESIGN_ONLY = True


## 1. Quantify the optimizer-state boundary


In [ ]:
TOTAL_PARAMETERS = 30_000_000_000
memory = {
    'bf16_weights_gib': TOTAL_PARAMETERS * 2 / 1024**3,
    'bf16_gradients_gib': TOTAL_PARAMETERS * 2 / 1024**3,
    'fp32_master_and_adam_moments_gib': TOTAL_PARAMETERS * 12 / 1024**3,
}
memory['subtotal_before_activations_gib'] = sum(memory.values())
print(json.dumps(memory, indent=2))


In [ ]:
from nemotron_ft_lab.hardware import inspect_cuda, validate_full_sft_hardware

inventory = inspect_cuda()
print(json.dumps(inventory.as_dict(), indent=2))
try:
    validate_full_sft_hardware(inventory)
    print('Full-SFT hardware gate passed; this notebook still does not launch it.')
except RuntimeError as exc:
    print('Design-only on this allocation:', exc)


## 2. Produce the external handoff command


In [ ]:
TARGET_GPUS = 16
full_cmd = [
    'torchrun', f'--nproc-per-node={TARGET_GPUS}', 'scripts/train_full.py',
    '--megatron-checkpoint', str(MEGATRON_BASE),
    '--data-dir', str(DATA_DIR), '--output-dir', str(FULL_ROOT),
    '--sequence-length', '2048', '--global-batch-size', '32',
    '--max-steps', '64', '--learning-rate', '5e-6',
]
assert DESIGN_ONLY
print('Command for a qualifying, separately rehearsed allocation:')
print(' '.join(full_cmd))
print('Trainable scope: every model parameter; no PEFT config.')


## 3. Require the same evidence after export


In [ ]:
required_evidence = {
    'base_checkpoint': str(MEGATRON_BASE),
    'training_data_manifest': str(DATA_DIR / 'training_manifest.json'),
    'frozen_local_baseline': str(BASELINE_PATH),
    'export_command': f'python scripts/export_full_checkpoint.py --megatron-checkpoint {FULL_ROOT} --output {FULL_HF}',
    'evaluation': 'run scripts/evaluate_vllm.py, then report paired execution-accuracy delta and 95% bootstrap CI',
    'topology': 'record GPU type/count, TP, EP, DP, software revisions, and cold/warm cache state',
}
print(json.dumps(required_evidence, indent=2))
print('No training, export, or evaluation was launched by this notebook.')


## Decision rule

Prefer LoRA when it meets the execution target. Escalate to full SFT only after
repeated controlled LoRA experiments show a material ceiling and the gain can
justify a separately operated multi-GPU training and serving lifecycle.
